 Fase 4 — Relatório Executivo Automático via LLM

Nesta etapa, conectamos as métricas de monitoramento (PSI, KS, AUC) calculadas na Fase 3
a um LLM (Claude), que gera um relatório estruturado interpretando os números.

**Técnica usada: Tool Use / Structured Output.** Em vez de simplesmente pedir para o
modelo "escrever um texto", forçamos ele a retornar um formato estruturado (JSON) com
campos fixos. Isso garante consistência entre relatórios e permite plugar o resultado
direto em um dashboard ou sistema de alertas — é assim que engenharia de LLM é usada
em produção, e não apenas em chatbots.


In [4]:
import os
os.chdir("/workspaces/monitor-credito-llm")
print("Diretório atual:", os.getcwd())

Diretório atual: /workspaces/monitor-credito-llm


In [5]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Carregando as métricas já calculadas na Fase 3
df_metricas = pd.read_csv("data/processed/metricas_mensais.csv")
df_metricas.head()


,mes,psi,ks,auc
0,1,0.006581,0.420958,0.784140
1,2,0.009625,0.369353,0.721103
2,3,0.014879,0.261122,0.657459
3,4,0.009494,0.343386,0.695287
4,5,0.009825,0.370187,0.731539


## 1. Definindo o schema do relatório

Esse é o "contrato" que força o LLM a responder sempre no mesmo formato.

In [6]:
TOOL_SCHEMA = {
    "name": "gerar_relatorio_risco",
    "description": "Gera um relatorio estruturado de monitoramento de modelo de credito",
    "input_schema": {
        "type": "object",
        "properties": {
            "nivel_alerta": {
                "type": "string",
                "enum": ["normal", "atencao", "critico"],
                "description": "Classificacao geral do risco nesta safra"
            },
            "resumo_executivo": {
                "type": "string",
                "description": "2-3 frases resumindo a situacao para um gestor nao-tecnico"
            },
            "causa_provavel": {
                "type": "string",
                "description": "Interpretacao tecnica do que os numeros sugerem"
            },
            "recomendacao": {
                "type": "string",
                "description": "Acao recomendada (ex: recalibrar, retreinar, monitorar, nenhuma acao)"
            }
        },
        "required": ["nivel_alerta", "resumo_executivo", "causa_provavel", "recomendacao"]
    }
}


## 2. Função que gera o relatório de um mês específico

In [7]:
SYSTEM_PROMPT = """Você é um analista sênior de risco de crédito, especialista em
monitoramento de modelos de score. Você escreve relatórios objetivos e técnicos,
seguindo boas práticas de mercado:

- PSI abaixo de 0.10 é estável; entre 0.10 e 0.25 merece atenção; acima de 0.25 é crítico
- Quede de AUC/KS indica perda de poder discriminante do modelo (problema mais sério que PSI alto sozinho)
- PSI alto com AUC/KS estáveis sugere mudança de população, não necessariamente falha do modelo
- Seja direto e evite jargão desnecessário no resumo executivo (ele é para um gestor, não um cientista de dados)
"""

def gerar_relatorio_mes(mes, df_metricas, historico_meses=3):
    linha_atual = df_metricas[df_metricas["mes"] == mes].iloc[0]
    historico = df_metricas[df_metricas["mes"] < mes].tail(historico_meses)

    contexto = f"""
Métricas do mês {mes}:
- PSI: {linha_atual['psi']:.3f}
- KS: {linha_atual['ks']:.3f}
- AUC: {linha_atual['auc']:.3f}

Histórico dos {historico_meses} meses anteriores:
{historico[['mes', 'psi', 'ks', 'auc']].to_string(index=False)}
"""

    resposta = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Gere o relatorio de monitoramento com base nestes dados:\n{contexto}"}],
        tools=[TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "gerar_relatorio_risco"},
    )

    bloco_tool = resposta.content[0]
    return bloco_tool.input


## 3. Testando em um mês estável (mês 6) e um mês de crise (mês 14)

In [8]:
relatorio_estavel = gerar_relatorio_mes(6, df_metricas)
print(json.dumps(relatorio_estavel, indent=2, ensure_ascii=False))


{
  "nivel_alerta": "normal",
  "resumo_executivo": "O modelo de score apresenta desempenho saudável e em melhora consistente no mês 6. O PSI de 0.022 indica que o perfil da população avaliada permanece estável em relação à base de desenvolvimento. O poder de discriminação do modelo — medido pelo KS (0.469) e AUC (0.788) — está não apenas dentro de parâmetros aceitáveis, mas em trajetória de crescimento contínuo nos últimos quatro meses. Nenhuma intervenção corretiva é necessária no momento.",
  "causa_provavel": "A estabilidade do PSI (abaixo de 0.10 em todos os meses monitorados, variando entre 0.009 e 0.022) descarta mudanças relevantes no perfil da população entrante. O crescimento consistente de KS (0.261 → 0.469) e AUC (0.657 → 0.788) ao longo dos meses 3 a 6 sugere que o modelo está ganhando poder preditivo à medida que a safra matura e mais comportamento de pagamento real se torna observável — padrão esperado em modelos com janela de performance ainda em consolidação. Não há si

In [9]:
relatorio_crise = gerar_relatorio_mes(14, df_metricas)
print(json.dumps(relatorio_crise, indent=2, ensure_ascii=False))


{
  "nivel_alerta": "critico",
  "resumo_executivo": "O modelo apresenta PSI extremamente elevado no mês 14 (1.091), muito acima do limiar crítico de 0.25, indicando uma mudança severa no perfil da população analisada em relação à base de desenvolvimento. Este padrão já havia se manifestado no mês 13 (PSI = 0.995), sinalizando que a ruptura populacional não é pontual, mas uma tendência consolidada. Em contrapartida, as métricas de desempenho discriminante (KS = 0.452 e AUC = 0.763) permanecem estáveis e em patamares saudáveis, sugerindo que o modelo ainda distingue bem bons e maus pagadores dentro da nova população — porém, os scores podem estar mal calibrados para o perfil atual dos clientes.",
  "causa_provavel": "O PSI crítico e persistente nos meses 13 e 14 (0.995 e 1.091, respectivamente) aponta para uma mudança estrutural significativa na composição da carteira ou no perfil dos solicitantes de crédito — possivelmente decorrente de alterações na política de originação, mudança de 

## Próximos passos

- [ ] Gerar relatórios para todos os 24 meses e salvar em `reports/`
- [ ] Montar um pequeno dashboard (Streamlit) mostrando os gráficos + o relatório do LLM lado a lado
- [ ] Testar o modelo pedindo para comparar dois meses específicos (ex: "por que o mês 14 foi pior que o mês 12?")
- [ ] Escrever o README final e o post do LinkedIn
